# PySpark & MLlib

## Fetching the Data

In [1]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh titanic
!ls -lh titanic/

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Réception d'objets: 100% (2/2), 2.96 Kio | 2.96 Mio/s, fait.
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 0), reused 5 (delta 0), pack-reused 0 (from 0)
Réception d'objets: 100% (5/5), 33.71 Kio | 1.87 Mio/s, fait.
✓ titanic — 104K
total 100K
-rw-rw-r-- 1 mog mog  237 11 nov.  22:30 data_description.txt
-rw-rw-r-- 1 mog mog 3,2K 11 nov.  22:30 gender_submission.csv
-rw-rw-r-- 1 mog mog  709 11 nov.  22:30 README.md
-rw-rw-r-- 1 mog mog  28K 11 nov.  22:30 test.csv
-rw-rw-r-- 1 mog mog  60K 11 nov.  22:30 train.csv


## Importing the Required Libraries

In [2]:
!pip install pyspark

pyenv: pip: command not found

The `pip' command exists in these Python versions:
  3.12.11
  3.13.7

Note: See 'pyenv help global' for tips on allowing both
      python2 and python3 to be found.


In [3]:
import re
from pathlib import Path

import pyspark.ml.feature as MF
import pyspark.sql.functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import (
  GBTClassifier,
  LinearSVC,
  LogisticRegression,
  RandomForestClassifier,
)
from pyspark.ml.evaluation import (
  BinaryClassificationEvaluator,
  MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession

## Loading the Data

The CSV file is loaded into a Spark `DataFrame`.

In [4]:
data_dir = Path("titanic")
csv_path = data_dir / "train.csv"

In [5]:
spark = SparkSession.builder.appName("PySpark demonstration").getOrCreate()
df = spark.read.csv(str(csv_path), header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/11 22:30:13 WARN Utils: Your hostname, pug, resolves to a loopback address: 127.0.1.1; using 192.168.1.94 instead (on interface wlo1)
26/11/11 22:30:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/11 22:30:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
passenger_count = df.count()
print(passenger_count)

891


### Partitions

A `DataFrame` is split into partitions, which the executors process in parallel, one task per partition. Here, everything runs in a single local process, whose cores play the part of the executors: `defaultParallelism` gives their number. The file, a small one, fits in a single partition.

In [7]:
print(f"Partitions: {df.rdd.getNumPartitions()}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

Partitions: 1
Default parallelism: 8


In [8]:
df.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [9]:
df.describe().show()

26/11/11 22:30:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 6:>                                                          (0 + 1) / 1]

+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|summary|      PassengerId|           Survived|            Pclass|                Name|   Sex|               Age|             SibSp|              Parch|            Ticket|             Fare|Cabin|Embarked|
+-------+-----------------+-------------------+------------------+--------------------+------+------------------+------------------+-------------------+------------------+-----------------+-----+--------+
|  count|              891|                891|               891|                 891|   891|               714|               891|                891|               891|              891|  204|     889|
|   mean|            446.0| 0.3838383838383838| 2.308641975308642|                NULL|  NULL| 29.69911764705882|0.5230078563411896|0.38159371492704824|260318.54916792738| 32.20420

In [10]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



## Renaming the Columns to Snake Case, Python's Convention

In [11]:
df = df.toDF(*(re.sub(r"(?<!^)(?=[A-Z])", "_", c).lower() for c in df.columns))
df.show(5)

+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+
|passenger_id|survived|pclass|                name|   sex| age|sib_sp|parch|          ticket|   fare|cabin|embarked|
+------------+--------+------+--------------------+------+----+------+-----+----------------+-------+-----+--------+
|           1|       0|     3|Braund, Mr. Owen ...|  male|22.0|     1|    0|       A/5 21171|   7.25| NULL|       S|
|           2|       1|     1|Cumings, Mrs. Joh...|female|38.0|     1|    0|        PC 17599|71.2833|  C85|       C|
|           3|       1|     3|Heikkinen, Miss. ...|female|26.0|     0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|           4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|     1|    0|          113803|   53.1| C123|       S|
|           5|       0|     3|Allen, Mr. Willia...|  male|35.0|     0|    0|          373450|   8.05| NULL|       S|
+------------+--------+------+--------------------+------+----+-

## Exploring the Data

In [12]:
df.select("survived", "pclass", "embarked").show(5)

+--------+------+--------+
|survived|pclass|embarked|
+--------+------+--------+
|       0|     3|       S|
|       1|     1|       C|
|       1|     3|       S|
|       1|     1|       S|
|       0|     3|       S|
+--------+------+--------+
only showing top 5 rows


In [13]:
df.groupBy("survived").count().show()

+--------+-----+
|survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



In [14]:
df.groupBy("sex", "survived").count().show()

+------+--------+-----+
|   sex|survived|count|
+------+--------+-----+
|  male|       0|  468|
|female|       1|  233|
|female|       0|   81|
|  male|       1|  109|
+------+--------+-----+



The same count in SQL: a temporary view gives the `DataFrame` a name, which the query refers to. Both versions produce the same execution plan.

In [15]:
df.createOrReplaceTempView("passengers")
spark.sql(
  "SELECT sex, survived, COUNT(*) AS count FROM passengers GROUP BY sex, survived"
).show()

+------+--------+-----+
|   sex|survived|count|
+------+--------+-----+
|  male|       0|  468|
|female|       1|  233|
|female|       0|   81|
|  male|       1|  109|
+------+--------+-----+



In [16]:
df.groupBy("pclass", "survived").count().show()

+------+--------+-----+
|pclass|survived|count|
+------+--------+-----+
|     1|       0|   80|
|     3|       1|  119|
|     1|       1|  136|
|     2|       1|   87|
|     2|       0|   97|
|     3|       0|  372|
+------+--------+-----+



## Preparing the Data

### Null Values

In [17]:
df.select(
  [F.count(F.when(F.col(column).isNull(), 1)).alias(column) for column in df.columns]
).show()

+------------+--------+------+----+---+---+------+-----+------+----+-----+--------+
|passenger_id|survived|pclass|name|sex|age|sib_sp|parch|ticket|fare|cabin|embarked|
+------------+--------+------+----+---+---+------+-----+------+----+-----+--------+
|           0|       0|     0|   0|  0|177|     0|    0|     0|   0|  687|       2|
+------------+--------+------+----+---+---+------+-----+------+----+-----+--------+



### Analyzing the Passengers' Age

In [18]:
mean_age = df.select(F.avg("age")).first()[0]
print(mean_age)

29.69911764705882


We could use this average age to impute the unknown ages, but we'll see how to go a bit further and compute the average age per title.

In [19]:
df = df.withColumn("title", F.regexp_extract(F.col("name"), r"([A-Za-z]+)\.", 1))
df.select("name", "sex", "title").show(5, truncate=False)

+---------------------------------------------------+------+-----+
|name                                               |sex   |title|
+---------------------------------------------------+------+-----+
|Braund, Mr. Owen Harris                            |male  |Mr   |
|Cumings, Mrs. John Bradley (Florence Briggs Thayer)|female|Mrs  |
|Heikkinen, Miss. Laina                             |female|Miss |
|Futrelle, Mrs. Jacques Heath (Lily May Peel)       |female|Mrs  |
|Allen, Mr. William Henry                           |male  |Mr   |
+---------------------------------------------------+------+-----+
only showing top 5 rows


In [20]:
df.select("title").distinct().show()

+--------+
|   title|
+--------+
|     Don|
|    Miss|
|Countess|
|     Col|
|     Rev|
|    Lady|
|  Master|
|     Mme|
|    Capt|
|      Mr|
|      Dr|
|     Mrs|
|     Sir|
|Jonkheer|
|    Mlle|
|   Major|
|      Ms|
+--------+



The rare titles are grouped together. "Dr" is used by men and by one woman: it goes to "Mr" or "Mrs" according to the sex.

In [21]:
groupings = dict(
  Miss=["Mlle", "Ms"],
  Mr=["Major", "Capt", "Sir", "Don"],
  Mrs=["Lady", "Countess", "Mme"],
  Other=["Jonkheer", "Col", "Rev"],
)
df = df.replace(
  {name: title for title, names in groupings.items() for name in names},
  subset=["title"],
)
df = df.withColumn(
  "title",
  F.when(
    F.col("title") == "Dr",
    F.when(F.col("sex") == "female", "Mrs").otherwise("Mr"),
  ).otherwise(F.col("title")),
)

In [22]:
df.groupBy("title").count().show()

+------+-----+
| title|count|
+------+-----+
|  Miss|  185|
| Other|    9|
|Master|   40|
|    Mr|  528|
|   Mrs|  129|
+------+-----+



### Adding a New Feature

The family size.

In [23]:
df = df.withColumn("family_size", F.col("sib_sp") + F.col("parch") + 1)

In [24]:
df.groupBy("family_size").count().show()

+-----------+-----+
|family_size|count|
+-----------+-----+
|          1|  537|
|          6|   22|
|          3|  102|
|          5|   15|
|          4|   29|
|          8|    6|
|          7|   12|
|         11|    7|
|          2|  161|
+-----------+-----+



In [25]:
df = df.withColumn("alone", (F.col("family_size") == 1).cast("long"))
df.select("sib_sp", "parch", "family_size", "alone").show(5)

+------+-----+-----------+-----+
|sib_sp|parch|family_size|alone|
+------+-----+-----------+-----+
|     1|    0|          2|    0|
|     1|    0|          2|    0|
|     0|    0|          1|    1|
|     1|    0|          2|    0|
|     0|    0|          1|    1|
+------+-----+-----------+-----+
only showing top 5 rows


### Transformations and Actions

None of the previous cells that modify `df` has read the file: `withColumn`, `replace` or `select` are *transformations*, which only add a step to the execution plan. Only *actions*, such as `show`, `count` or `collect`, start a computation, and Spark then optimizes the whole plan at once. `explain` prints the physical plan it chose: the CSV scan (`FileScan csv`), then the computed columns (`Project`), all marked `*(1)`: Spark runs them in a single pass over the data.

In [26]:
df.explain()

== Physical Plan ==
*(1) Project [passenger_id#929, survived#930, pclass#931, name#932, sex#933, age#934, sib_sp#935, parch#936, ticket#937, fare#938, cabin#939, embarked#940, title#1250, family_size#1277, cast((family_size#1277 = 1) as bigint) AS alone#1305L]
+- *(1) Project [passenger_id#929, survived#930, pclass#931, name#932, sex#933, age#934, sib_sp#935, parch#936, ticket#937, fare#938, cabin#939, embarked#940, CASE WHEN (title#1249 = Dr) THEN CASE WHEN (sex#933 = female) THEN Mrs ELSE Mr END ELSE title#1249 END AS title#1250, ((sib_sp#935 + parch#936) + 1) AS family_size#1277]
   +- *(1) Project [passenger_id#929, survived#930, pclass#931, name#932, sex#933, age#934, sib_sp#935, parch#936, ticket#937, fare#938, cabin#939, embarked#940, CASE WHEN (title#1231 = Don) THEN Mr WHEN (title#1231 = Col) THEN Other WHEN (title#1231 = Mme) THEN Mrs WHEN (title#1231 = Sir) THEN Mr WHEN (title#1231 = Jonkheer) THEN Other WHEN (title#1231 = Ms) THEN Miss WHEN (title#1231 = Countess) THEN Mrs 

### Splitting Into Training and Test Sets

The columns computed so far depend only on the passenger's own row. What comes next learns from the data: the average age per title, the most frequent port, then the categories the encoders keep. So we split the data now, and everything learned will be learned on the training set alone: computed on all 891 passengers, it would feed the model information about the test passengers (a data leak).

In [27]:
training, test = df.randomSplit([0.8, 0.2], seed=11)
print(f"Training: {training.count()}, test: {test.count()}")

Training: 703, test: 188


### Average Age per Title

In [28]:
average_ages = training.groupBy("title").agg(F.avg("age").alias("average_age"))
average_ages.show()

+------+------------------+
| title|       average_age|
+------+------------------+
|  Miss|21.743589743589745|
| Other| 42.42857142857143|
|Master|4.5125925925925925|
|    Mr|32.458730158730155|
|   Mrs| 35.54455445544554|
+------+------------------+



### Most Frequent Port of Embarkation

The *embarked* feature (the port of embarkation) has only two missing values: they get the most frequent port of the training set.

In [29]:
training.groupBy("embarked").count().show()

+--------+-----+
|embarked|count|
+--------+-----+
|       Q|   60|
|    NULL|    2|
|       C|  137|
|       S|  504|
+--------+-----+



In [30]:
most_frequent_port = (
  training.dropna(subset=["embarked"])
  .groupBy("embarked")
  .count()
  .orderBy(F.desc("count"))
  .first()["embarked"]
)
print(most_frequent_port)

S


### Imputation

Each passenger whose age is unknown gets the average age of the training passengers sharing their title. The `left` join keeps every passenger, even one whose title is absent from the training set, and `coalesce` only replaces the null ages.

The *cabin* feature is dropped: the cabin number is almost unique for each passenger in a cabin, and would quickly lead to overfitting.

In [31]:
def complete(data):
  return (
    data.join(average_ages, "title", "left")
    .withColumn("age", F.coalesce("age", "average_age"))
    .drop("average_age", "cabin")
    .fillna({"embarked": most_frequent_port})
  )


training = complete(training)
test = complete(test)
training.select("name", "title", "age", "embarked").show(5)

+--------------------+-----+----+--------+
|                name|title| age|embarked|
+--------------------+-----+----+--------+
|Braund, Mr. Owen ...|   Mr|22.0|       S|
|Cumings, Mrs. Joh...|  Mrs|38.0|       C|
|Heikkinen, Miss. ...| Miss|26.0|       S|
|Futrelle, Mrs. Ja...|  Mrs|35.0|       S|
|Allen, Mr. Willia...|   Mr|35.0|       S|
+--------------------+-----+----+--------+
only showing top 5 rows


The four models and the cross-validation read the training set again and again: without `cache()`, each fit would read the CSV again and replay the regular expression, the split and the join. Like a transformation, `cache()` computes nothing: the data is kept in memory at the first action.

In [32]:
training = training.cache()

## Machine Learning with MLlib (Optional)

### The Pipeline

A `Pipeline` chains the preparation steps and the model. Its `fit` fits each step on the training set, in order, and returns a `PipelineModel`, whose `transform` applies the same steps, already fitted, to other data:

- `StringIndexer` replaces each category with an integer;
- `OneHotEncoder` replaces that integer with indicator columns (one 0 or 1 column per category), so that a linear model doesn't read an order between the ports or the titles;
- `VectorAssembler` gathers the features into a single vector, the column MLlib's models expect;
- the classifier.

In [33]:
categorical = ["sex", "embarked", "title"]
numerical = ["pclass", "age", "sib_sp", "parch", "fare", "family_size", "alone"]


def build_pipeline(classifier):
  return Pipeline(
    stages=[
      MF.StringIndexer(
        inputCols=categorical,
        outputCols=[f"{column}_index" for column in categorical],
      ),
      MF.OneHotEncoder(
        inputCols=[f"{column}_index" for column in categorical],
        outputCols=[f"{column}_onehot" for column in categorical],
      ),
      MF.VectorAssembler(
        inputCols=numerical + [f"{column}_onehot" for column in categorical],
        outputCol="features",
      ),
      classifier,
    ]
  )

### Evaluation

Two measures on the test set: the accuracy, the share of passengers correctly classified, and the area under the ROC curve (AUC), which judges how the scores rank the passengers, whatever the decision threshold. The baseline is the accuracy of the majority class: predicting that nobody survives.

In [34]:
accuracy = MulticlassClassificationEvaluator(
  labelCol="survived", predictionCol="prediction", metricName="accuracy"
)
auc = BinaryClassificationEvaluator(labelCol="survived")


def evaluate(model, name):
  predictions = model.transform(test)
  print(
    f"{name}: accuracy {accuracy.evaluate(predictions):.2f},"
    f" AUC {auc.evaluate(predictions):.2f}"
  )


test_count = test.count()
death_count = test.where(F.col("survived") == 0).count()
print(
  f"Majority class: accuracy {death_count / test_count:.2f} on {test_count} passengers"
)

Majority class: accuracy 0.63 on 188 passengers


### Logistic Regression

In [35]:
logistic_model = build_pipeline(
  LogisticRegression(labelCol="survived", featuresCol="features")
).fit(training)
evaluate(logistic_model, "Logistic regression")
logistic_model.transform(test).select(
  "name", "survived", "prediction", "probability"
).show(5)

Logistic regression: accuracy 0.82, AUC 0.86
+--------------------+--------+----------+--------------------+
|                name|survived|prediction|         probability|
+--------------------+--------+----------+--------------------+
|    Moran, Mr. James|       0|       0.0|[0.90317896523785...|
|McCarthy, Mr. Tim...|       0|       0.0|[0.70104875409283...|
|Saundercock, Mr. ...|       0|       0.0|[0.90439068452074...|
|Andersson, Mr. An...|       0|       0.0|[0.99272352209283...|
|Vestrom, Miss. Hu...|       0|       1.0|[0.31592575490621...|
+--------------------+--------+----------+--------------------+
only showing top 5 rows


### Random Forest

In [36]:
forest_model = build_pipeline(
  RandomForestClassifier(labelCol="survived", featuresCol="features")
).fit(training)
evaluate(forest_model, "Random forest")

Random forest: accuracy 0.83, AUC 0.87


### Gradient-Boosted Trees

In [37]:
gbt_model = build_pipeline(
  GBTClassifier(labelCol="survived", featuresCol="features", maxIter=10)
).fit(training)
evaluate(gbt_model, "Gradient-boosted trees")

Gradient-boosted trees: accuracy 0.80, AUC 0.83


### Support Vector Machine

In [38]:
svm_model = build_pipeline(LinearSVC(labelCol="survived", featuresCol="features")).fit(
  training
)
evaluate(svm_model, "Support vector machine")

26/11/11 22:31:05 ERROR OWLQN: Failure! Resetting history: breeze.optimize.NaNHistory: 


Support vector machine: accuracy 0.82, AUC 0.86


### Tuning With Cross-Validation

`CrossValidator` chooses the hyperparameters on the training set alone: it splits it into three, fits the pipeline on two thirds and measures it on the third one, in turn, for each combination of the grid `ParamGridBuilder` builds, then refits the best one on the whole training set. The test set thus stays out of the choice. Here, the grid covers the regularization of the logistic regression.

In [39]:
logistic = LogisticRegression(labelCol="survived", featuresCol="features")
grid = ParamGridBuilder().addGrid(logistic.regParam, [0.0, 0.1, 1.0]).build()
validation = CrossValidator(
  estimator=build_pipeline(logistic),
  estimatorParamMaps=grid,
  evaluator=auc,
  numFolds=3,
  seed=11,
)
validated_model = validation.fit(training)
for params, mean_auc in zip(grid, validated_model.avgMetrics):
  print(f"regParam = {params[logistic.regParam]}: AUC {mean_auc:.3f}")
best_model = validated_model.bestModel
evaluate(best_model, "Tuned logistic regression")

regParam = 0.0: AUC 0.873
regParam = 0.1: AUC 0.870
regParam = 1.0: AUC 0.857
Tuned logistic regression: accuracy 0.82, AUC 0.86


### Conclusion

The four models clearly beat the majority class (0.63): from 0.80 to 0.84 accuracy, from 0.83 to 0.88 AUC. Their differences, though, prove nothing. The random forest's score itself varies from one run to the next (from 0.82 to 0.84 accuracy in our runs), and the test set holds only 188 passengers: each is worth a little over half a point of accuracy, and the margin of error of a 0.82 accuracy measured on 188 passengers is about 5 points either way. Another split (another `randomSplit` seed) could change the ranking. To choose between models or hyperparameters, we therefore compare their cross-validation scores, on the training set, as `CrossValidator` does above; the test set only measures the chosen model, once. Here, cross-validation prefers the logistic regression without regularization, the default setting: the tuned model is the first one.

## Saving and Reusing the Model

A `PipelineModel` is saved with all its fitted steps (the encoders' categories, the model's coefficients): another Spark program loads it and applies it to new data, without learning anything again. This is how a model goes into production in a batch job.

In [40]:
model_path = "titanic_model"
best_model.write().overwrite().save(model_path)
loaded_model = PipelineModel.load(model_path)

Three made-up passengers, described by the original columns and their title; the family size and `alone` are computed as above, and the loaded model does the rest:

In [41]:
new_passengers = spark.createDataFrame(
  [
    ("female", 1, 35.0, 1, 0, 80.0, "C", "Mrs"),
    ("male", 3, 28.0, 0, 0, 8.0, "S", "Mr"),
    ("male", 2, 6.0, 1, 1, 30.0, "S", "Master"),
  ],
  ["sex", "pclass", "age", "sib_sp", "parch", "fare", "embarked", "title"],
)
new_passengers = new_passengers.withColumn(
  "family_size", F.col("sib_sp") + F.col("parch") + 1
).withColumn("alone", (F.col("family_size") == 1).cast("long"))
loaded_model.transform(new_passengers).select(
  "sex", "pclass", "age", "title", "probability", "prediction"
).show(truncate=False)

+------+------+----+------+----------------------------------------+----------+
|sex   |pclass|age |title |probability                             |prediction|
+------+------+----+------+----------------------------------------+----------+
|female|1     |35.0|Mrs   |[0.03789324277389312,0.9621067572261068]|1.0       |
|male  |3     |28.0|Mr    |[0.9213895603024594,0.07861043969754056]|0.0       |
|male  |2     |6.0 |Master|[0.13784107749083357,0.8621589225091664]|1.0       |
+------+------+----+------+----------------------------------------+----------+



## The Same Code on a Cluster

Nothing in this notebook depends on the local mode, except the session. On a cluster, it is created with the address of its manager, for instance `SparkSession.builder.master("spark://host:7077")` for a standalone Spark cluster, or `spark-submit --master yarn` launches the script; the paths then point to a shared storage (HDFS, S3…), readable by every executor. The partitions are spread across the machines, and the steps that bring together rows from different partitions (a `groupBy`, fitting a model) move data between them: that's the `Exchange` step `explain` prints for a `groupBy`. The code of the transformations, of the pipeline and of the saving doesn't change.

## Reference

https://spark.apache.org/docs/latest/ml-classification-regression.html